# Pydantic的使用

## 1、基本使用

举例1：


In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

from langchain_community.chat_models import ChatZhipuAI
from langchain_openai import ChatOpenAI
from pypdf.constants import FieldDictionaryAttributes
from sympy.physics import pring

load_dotenv(override=True)

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL"),
    #  extra_body={
    #     "thinking": {
    #         "type": "disabled"
    #     }
    # }
)


In [5]:
from pydantic import BaseModel, Field


#定义结构类型
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")


#把定义好的结构类型和模型关联起来，通过with_structured_output来进行关联,关联完以后得得到一个新的结果
structured_model = model.with_structured_output(Person)
#然后用这个新的结果再去调用invoke
result = structured_model.invoke("张三是一名30岁的软件工程师")

print(result)
print(type(result))

name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [6]:
print(f"姓名：{result.name}")
print(f"年龄：{result.age}")
print(f"职业：{result.occupation}")

姓名：张三
年龄：30
职业：软件工程师


举例2：

In [7]:

class MovieModel(BaseModel):
    """电影的详细信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="发行年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")


structured_model = model.with_structured_output(MovieModel)

result = structured_model.invoke("给出电影盗梦空间的信息")
print(result)

title='电影《盗梦空间》(Inception) 详细信息' year=2010 director='克里斯托弗·诺兰 (Christopher Nolan)' rating=8.8


举例3:

In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field

load_dotenv(override=True)

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL"),
)


#定义结构类型
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0~1之间")
    keywords: list[str] = Field(description="关键词列表")


structured_model = model.with_structured_output(SentimentAnalysis)

text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = structured_model.invoke(f"分析以下文本的情感：\n{text}")

print(f"类型:{type(result)}")
print(f"情感:{result.sentiment}")
print(f"置信度:{result.confidence}")
print(f"关键词:{result.keywords}")


类型:<class '__main__.SentimentAnalysis'>
情感:正面
置信度:0.99
关键词:['实用', '学到', '强烈推荐']


## 2、高级特性


### 2.1 情况1：可选字段

举例：

Deepseek模型

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override=True)
model_deepseek = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

智谱AI模型

In [57]:
from dotenv import load_dotenv
from langchain_community.chat_models.zhipuai import ChatZhipuAI
import os

load_dotenv(override=True)
model_zhipu = ChatZhipuAI(
    model="glm-5.2",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
     extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

model_zhipu.invoke("用一句话介绍一下你自己")

AIMessage(content='我是Z.ai训练的大型语言模型，旨在通过自然对话的方式为您提供信息、解答问题并协助完成各种任务。', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 297, 'completion_tokens_details': {'reasoning_tokens': 270}, 'prompt_tokens': 16, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 313}, 'model_name': 'glm-5.2', 'finish_reason': 'stop'}, id='lc_run--01a01912-4874-7392-b4b1-65f1d4808a54-0', tool_calls=[], invalid_tool_calls=[])

In [11]:
from pydantic import BaseModel, Field


#定义结构类型
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")


#把定义好的结构类型和模型关联起来，通过with_structured_output来进行关联,关联完以后得得到一个新的结果
structured_model = model_deepseek.with_structured_output(Person)
#然后用这个新的结果再去调用invoke
result = structured_model.invoke("张三是一名律师")

print(result)
print(type(result))

name='张三' age=0 occupation='律师'
<class '__main__.Person'>


作为对比：

In [12]:
from typing import Optional
from pydantic import BaseModel, Field


#定义结构类型
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")


#把定义好的结构类型和模型关联起来，通过with_structured_output来进行关联,关联完以后得得到一个新的结果
structured_model = model_deepseek.with_structured_output(Person)
#然后用这个新的结果再去调用invoke
result = structured_model.invoke("张三是一名律师")

print(result)
print(type(result))

name='张三' age=None occupation='律师'
<class '__main__.Person'>


### 2.2 情况2：默认值

不同的模型供应商，对于这个默认值字段的支持是不同的。比如：closeai平台的gpt-5.4-mini模型就不支持此字段
而openrouter平台的gpt-5.4-mini模型就支持此字段

下面使用的是国内的两个不同的模型deepseek和月之暗面

举例

In [13]:
from pydantic import BaseModel, Field


#定义结构类型
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(default=10, description="年龄")
    occupation: str = Field(description="职业")


#把定义好的结构类型和模型关联起来，通过with_structured_output来进行关联,关联完以后得得到一个新的结果
structured_model = model_deepseek.with_structured_output(Person)
#然后用这个新的结果再去调用invoke
result = structured_model.invoke("张三是一名律师")

print(result)
print(type(result))

name='张三' age=10 occupation='律师'
<class '__main__.Person'>


可以看到上面的模型deepseek是支持default的  作为对比：换一个模型供应商  结果：可以看出设置了默认值，但是输出并没有使用默认值，所以这个模型
供应商是不支持default字段的

In [51]:
from pydantic import BaseModel, Field


#定义结构类型
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(default=10, description="年龄")
    occupation: str = Field(description="职业")


#把定义好的结构类型和模型关联起来，通过with_structured_output来进行关联,关联完以后得得到一个新的结果
structured_model = model_zhipu.with_structured_output(Person)
#然后用这个新的结果再去调用invoke
result = structured_model.invoke("张三是一名律师")

print(result)
print(type(result))

name='张三' age=10 occupation='律师'
<class '__main__.Person'>


举例2：


In [15]:
from pydantic import BaseModel, Field


class Config(BaseModel):
    timeout: Optional[int] = Field(default=30, description="超市时间(单位秒)")
    retry: bool = Field(default=False, description="是否支持重试")
    max_attempts: int = Field(default=6, description="最大重试次数")


structured_model = model_deepseek.with_structured_output(Config)
structured_model.invoke("配置要求：支持重试，最多重试5次")

Config(timeout=30, retry=True, max_attempts=5)

最为对比：换月之暗面模型

In [53]:
from pydantic import BaseModel, Field
from typing import Optional


class Config(BaseModel):
    timeout: Optional[int] = Field(default=30, description="超市时间(单位秒)")
    retry: bool = Field(default=False, description="是否支持重试")
    max_attempts: int = Field(default=6, description="最大重试次数")


structured_model = model_zhipu.with_structured_output(Config)
structured_model.invoke("配置要求：支持重试，最多重试5次")


Config(timeout=30, retry=True, max_attempts=5)

举例3：

In [23]:
from typing import Optional
from pydantic import BaseModel, Field


class Product(BaseModel):
    """产品信息"""
    name: str = Field(description="产品名称")
    price: float = Field(description="价格")
    description: Optional[str] = Field(description="产品描述")
    stock: int = Field(default=100, description="库存")


structured_model = model_deepseek.with_structured_output(Product)
print("\n场景1：完整信息")
result = structured_model.invoke("iPhone16 售价 5999元，最新款智能手机，库存50台")
print(result)

print("\n场景2：缺少描述和库存")
result2 = structured_model.invoke("MacBook Pro 售价 12999元")
print(result2)


场景1：完整信息
name='iPhone16' price=5999.0 description='最新款智能手机' stock=50

场景2：缺少描述和库存
name='MacBook Pro' price=12999.0 description='MacBook Pro' stock=100


### 2.3 情况3：枚举类型

方式1：

In [7]:
from enum import Enum
from pydantic import BaseModel, Field
from typing import Optional


#定义枚举类型
class Priority(str, Enum):
    Low = "低"
    MEDIUM = "中"
    HIGH = "高"


#定义结构化输出
class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")


structured_model = model_deepseek.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您?
客户: 我是王小明，电话138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_model.invoke(f"从以下客服对话中提取客户信息{conversation}")

print(result)

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency='高'


方式2：

In [8]:
from typing import Literal


#定义结构化输出
class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    #在这里直接使用Literal对枚举进行定义
    urgency: Literal["低", "中", "高"] = Field(description="紧急程度")


structured_model = model_deepseek.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您?
客户: 我是王小明，电话138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_model.invoke(f"从以下客服对话中提取客户信息{conversation}")

print(result)

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency='高'


### 2.4 情况4：列表提取

举例1：

In [3]:
from typing import List
from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")


class PersonList(BaseModel):
    people: List[Person]


structured_model = model_deepseek.with_structured_output(PersonList)

result = structured_model.invoke("张三 30岁,李四 40岁")

print(result)

people=[Person(name='张三', age=30), Person(name='李四', age=40)]


举例2：

In [4]:
class Review(BaseModel):
    """产品评论"""
    product: str = Field(description="产品名称")
    rating: int = Field(description="评分1-5")
    pros: List[str] = Field(description="忧点列表")
    cons: List[str] = Field(description="缺点列表")


structured_model = model_deepseek.with_structured_output(Review)

result = structured_model.invoke("iPhone17 很棒！ 摄像头强大，手感好。但是价格贵，没有充电器。4分")

print(result)


product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


举例3：

In [5]:
class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: List[str] = Field(description="商品")


structured_model = model_deepseek.with_structured_output(Invoice)
invoice_text = ("发票号：INV-2024-001"
                "日期：2024-1-15"
                "总金额：1299.00"
                "商品：MacBook Pro,AppleCare+")

result = structured_model.invoke(f"提取发票信息：{invoice_text}")
print(result)

invoice_number='INV-2024-001' date='2024-1-15' total_amount=1299.0 items=['MacBook Pro', 'AppleCare+']


### 2.5 情况5：嵌套结构

举例1：

In [6]:
class Address(BaseModel):
    """地点描述"""
    city: str = Field(description="城市")
    district: str = Field(description="区域")


class Company(BaseModel):
    """公司信息"""
    name: str = Field(description="公司名称")
    address: Address = Field(description="公司所在地")


structured_model = model_deepseek.with_structured_output(Company)

result = structured_model.invoke("阿里巴巴在杭州的滨江区")

print(result)

name='阿里巴巴' address=Address(city='杭州', district='滨江区')


举例2:

In [8]:
from pydantic import BaseModel, Field
from typing import List


# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表")  # 定义列表字段
    rating: float = Field(description="评分")


# 2. 初始化模型并绑定输出结构
structured_model = model_deepseek.with_structured_output(Movie)
# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke("请介绍电影《盗梦空间》")
# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")


电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表: [Actor(name='莱昂纳多·迪卡普里奥', role='道姆·柯布'), Actor(name='约瑟夫·高登-莱维特', role='亚瑟'), Actor(name='艾伦·佩吉', role='阿里阿德涅'), Actor(name='汤姆·哈迪', role='伊姆斯'), Actor(name='玛丽昂·歌迪亚', role='梅尔'), Actor(name='渡边谦', role='斋藤'), Actor(name='基里安·墨菲', role='罗伯特·费舍')]
评分: 9.4


举例3：

In [14]:
from pydantic import BaseModel
from typing import List


class Aspect(BaseModel):
    """评论维度"""
    name: str = Field(description="维度名称，如：质量、价格、服务")
    score: int = Field(description="评分，1-5")
    comment: str = Field(description="具体评价")


class ProductReview(BaseModel):
    """产品评论分析"""
    overall_sentiment: str = Field(description="整体情感：positive/negative/neutral")
    overall_score: int = Field(description="综合评分，1-5")
    aspects: List[Aspect] = Field(description="各维度评价")
    summary: str = Field(description="一句话总结")


# 创建结构化模型
structured_model = model_deepseek.with_structured_output(ProductReview)
# 测试
review_text = """
这款笔记本电脑性能非常强大，运行大型软件毫无压力。
屏幕色彩鲜艳，看视频很舒服。
不过价格有点贵，而且风扇噪音较大。
客服态度很好，物流也快。
总体来说还是值得购买的。
"""
result = structured_model.invoke(
    f"分析以下产品评论：\n{review_text}"
)
print(f"整体情感: {result.overall_sentiment}")
print(f"综合评分: {result.overall_score}/5")
print(f"\n各维度评价:")
for aspect in result.aspects:
    print(f"  - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
print(f"\n总结: {result.summary}")


整体情感: positive
综合评分: 4/5

各维度评价:
  - 性能: 5/5 - 性能非常强大，运行大型软件毫无压力
  - 屏幕: 5/5 - 屏幕色彩鲜艳，看视频很舒服
  - 价格: 3/5 - 价格有点贵
  - 噪音: 3/5 - 风扇噪音较大
  - 服务: 5/5 - 客服态度很好，物流也快

总结: 产品性能和屏幕表现出色，服务好，但价格偏贵且风扇噪音较大，总体仍值得购买。


### 2.6 情况6：限制条件

In [20]:
from pydantic import ValidationError


class User(BaseModel):
    """"""
    name: str = Field(description="姓名", min_length=2, max_length=50)  #来设置当前字段在赋值的时候名称的长短
    age: int = Field(description="年龄", le=150)  #同样也是用来设置当前这个字段的长短，这里是年龄，设置的是年龄小于等于150岁
    email: str = Field(description="邮箱")


try:
    user1 = User(name="tom", age=22, email="tom@126.com")
    print(f"[OK]{user1}")
except ValidationError as e:
    print(f"[FAIL]{e}")

[OK]name='tom' age=22 email='tom@126.com'


In [59]:
from pydantic import ValidationError

try:
    user2 = User(name="tom", age=220, email="tom@126.com")
    print(f"[OK]{user2}")
except ValidationError as e:
    print(f"[FAIL]{e}")

[FAIL]1 validation error for User
age
  Input should be less than or equal to 150 [type=less_than_equal, input_value=220, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal


举例1：

注意：不同的模型供应商使用的效果不一样，这里使用的是DeepSeek

In [60]:
from pydantic import ValidationError
class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)
# 测试
structured_llm = model_deepseek.with_structured_output(Product)
response = structured_llm.invoke("华为mate 80 promax 价格是7999，当前库存100")
# response = structured_llm.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
print(response)

name='华为mate 80 promax' price=7999.0 stock=100


In [61]:
response = structured_llm.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
print(response)

ValidationError: 2 validation errors for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-7999, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than
stock
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than_equal

换智谱AI

In [58]:
from pydantic import ValidationError

class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)
# 测试
structured_llm = model_zhipu.with_structured_output(Product)
response = structured_llm.invoke("华为mate 80 promax 价格是7999，当前库存100")
# response = structured_llm.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
print(response)

name='华为mate 80 promax' price=7999.0 stock=100


In [55]:
response = structured_llm.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
print(response)

None
